# Transition-Based Dependency Parsing

Wiki reference for [dependency parsing](https://ml-viz-ruby.vercel.app/wiki/dependency-parsing).

**The idea in one sentence.** The **arc-standard** parser builds a dependency tree by consuming
a sentence left-to-right with a stack and a buffer, applying `SHIFT` / `LEFT-ARC` / `RIGHT-ARC`
transitions — a linear-time greedy procedure whose accuracy is measured by **UAS** (fraction of
tokens with the correct head), and whose weakness is that an early wrong action cascades.

We implement the arc-standard parser and UAS from scratch, **validate that the oracle produces
the gold tree and that UAS scores it**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0', 'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748', 'grid.color': '#2d3748', 'axes.grid': False,
})

## 1. The arc-standard transition system

Configuration = (stack, buffer, arcs). Start: stack=[ROOT], buffer=[all words]. Goal: buffer empty, stack=[ROOT].

In [ ]:
def arc_standard_parse(words, oracle_actions, verbose=True):
    """
    Run the arc-standard parser given a list of oracle actions.
    words: list of tokens (index 0 = ROOT)
    Returns arcs as list of (head_idx, dependent_idx).
    """
    stack = [0]                 # ROOT
    buffer = list(range(1, len(words)))
    arcs = []

    def fmt(idxs): return '[' + ', '.join(words[i] for i in idxs) + ']'

    for action in oracle_actions:
        if verbose:
            print(f"stack={fmt(stack):35s} buffer={fmt(buffer):30s} → {action}")
        if action == 'SHIFT':
            stack.append(buffer.pop(0))
        elif action == 'LEFT-ARC':
            # top is head of second-top; remove second-top
            dependent = stack[-2]
            head = stack[-1]
            arcs.append((head, dependent))
            del stack[-2]
        elif action == 'RIGHT-ARC':
            # second-top is head of top; remove top
            dependent = stack[-1]
            head = stack[-2]
            arcs.append((head, dependent))
            stack.pop()
    return arcs

# Parse: "The cat sat"  (ROOT The cat sat)
words = ['ROOT', 'The', 'cat', 'sat']
# Oracle: build  The<-cat, cat<-sat (subj), sat<-ROOT
oracle = ['SHIFT', 'SHIFT', 'LEFT-ARC', 'SHIFT', 'LEFT-ARC', 'RIGHT-ARC']
arcs = arc_standard_parse(words, oracle)
print("\nFinal arcs (head → dependent):")
for h, d in arcs:
    print(f"  {words[h]} → {words[d]}")

### Validate: the parser builds a well-formed tree

An arc-standard parse of an $n$-token sentence (including ROOT) produces exactly $n-1$ arcs —
one head per non-ROOT token — i.e. a valid tree. We confirm the arc count and that every
dependent gets exactly one head.

In [ ]:
deps = sorted(d for h, d in arcs)
print(f'{len(words)} tokens (incl. ROOT), {len(arcs)} arcs; dependents = {deps}')
assert len(arcs) == len(words) - 1, 'arc-standard produces one arc per non-ROOT token (a tree)'
assert deps == list(range(1, len(words))), 'every non-ROOT token is a dependent exactly once (one head each)'
print('\n✅ SHIFT/LEFT-ARC/RIGHT-ARC transitions assemble a well-formed dependency tree')

## 2. Visualize the dependency tree

In [ ]:
def plot_dependency_tree(words, arcs, title='Dependency parse'):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    x = np.arange(len(words))
    ax.scatter(x, np.zeros_like(x), s=10, color='#6366f1')
    for i, w in enumerate(words):
        ax.text(i, -0.08, w, ha='center', va='top', color='#e2e8f0', fontsize=11)
    for h, d in arcs:
        # Draw an arc from head to dependent
        mid = (h + d) / 2
        height = 0.15 + 0.15 * abs(h - d)
        xs = np.linspace(h, d, 50)
        ys = height * np.sin(np.pi * (xs - h) / (d - h + 1e-9))
        ax.plot(xs, ys, color='#22d3ee', linewidth=1.5)
        ax.annotate('', xy=(d, 0.02), xytext=(d, ys[len(ys)//2]),
                    arrowprops=dict(arrowstyle='->', color='#22d3ee'))
    ax.set_ylim(-0.3, 1.0)
    ax.set_xlim(-0.5, len(words)-0.5)
    ax.axis('off')
    ax.set_title(title, color='#e2e8f0')
    plt.tight_layout()
    plt.show()

plot_dependency_tree(words, arcs, 'Parse of "The cat sat"')

## 3. UAS evaluation

Unlabeled Attachment Score = fraction of words assigned the correct head.

In [ ]:
def arcs_to_heads(arcs, n_words):
    """Convert arc list to head array: head[d] = h."""
    heads = [-1] * n_words
    for h, d in arcs:
        heads[d] = h
    return heads

def uas(pred_arcs, gold_arcs, n_words):
    pred_heads = arcs_to_heads(pred_arcs, n_words)
    gold_heads = arcs_to_heads(gold_arcs, n_words)
    # exclude ROOT (index 0)
    correct = sum(1 for d in range(1, n_words) if pred_heads[d] == gold_heads[d])
    return correct / (n_words - 1)

gold_arcs = arcs  # our oracle parse is the gold standard here
# A wrong parse: attach 'cat' to ROOT instead of 'sat'
wrong_arcs = [(2, 1), (0, 2), (0, 3)]
print(f"UAS (perfect parse): {uas(arcs, gold_arcs, len(words)):.3f}")
print(f"UAS (wrong parse):   {uas(wrong_arcs, gold_arcs, len(words)):.3f}")

### Validate: UAS scores the parse

Unlabeled Attachment Score is the fraction of non-ROOT tokens assigned their correct head. The
oracle parse matches the gold tree (UAS = 1), while a wrong attachment scores below 1. We
confirm.

In [ ]:
print(f'UAS(oracle vs gold) = {uas(arcs, gold_arcs, len(words)):.3f}')
print(f'UAS(wrong  vs gold) = {uas(wrong_arcs, gold_arcs, len(words)):.3f}')
assert uas(arcs, gold_arcs, len(words)) == 1.0, 'the oracle reproduces the gold tree (UAS=1)'
assert uas(wrong_arcs, gold_arcs, len(words)) < 1.0, 'a wrong head attachment lowers UAS'
print('\n✅ UAS = fraction of tokens with the correct head — the standard parsing metric')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **error propagation** | one early wrong action cascades (demo) — use beam search |
| **projectivity** | arc-standard can't produce crossing (non-projective) arcs |
| **greedy vs graph-based** | linear-time greedy trades accuracy for speed |
| **oracle quality** | training needs a correct action oracle for each gold tree |
| **UAS vs LAS** | UAS ignores arc labels; LAS also checks the relation |

Demo: a wrong first action breaks the whole greedy parse.

In [ ]:
# Greedy transition parsing's Achilles heel: ERROR PROPAGATION. There is no backtracking, so a
# single wrong action early on corrupts every subsequent state — the parse can't recover, and
# may even become ill-formed. We feed a wrong FIRST action and confirm the result is no longer
# the gold parse.
corrupt = ['LEFT-ARC'] + oracle[1:]     # a wrong first action
ok = True
try:
    bad = arc_standard_parse(words, corrupt, verbose=False)
    ok = (uas(bad, gold_arcs, len(words)) == 1.0)
except Exception:
    ok = False                           # a wrong action can even make the parse ill-formed
assert not ok, 'one wrong early action breaks the greedy parse — no backtracking'
print('A single early mistake cascades -> beam search / graph-based parsers mitigate error propagation.')

## ✏️ Your turn

**Exercise 1 — Parse a longer sentence.** Find an oracle action sequence that parses `ROOT she ate pizza` into: `she <- ate` (subject), `pizza <- ate` (object), `ate <- ROOT`. Run the parser and verify UAS = 1.0 against the gold arcs `[(2,1),(2,3),(0,2)]`.

In [ ]:
words2 = ['ROOT', 'she', 'ate', 'pizza']
gold2 = [(2,1), (2,3), (0,2)]
# TODO(you): find the oracle action list and run arc_standard_parse(words2, oracle2)
# oracle2 = [...]

In [ ]:
# Assert cell
oracle2_ref = ['SHIFT', 'SHIFT', 'LEFT-ARC', 'SHIFT', 'RIGHT-ARC', 'RIGHT-ARC']
arcs2 = arc_standard_parse(words2, oracle2_ref, verbose=False)
score = uas(arcs2, gold2, len(words2))
print(f"Predicted arcs: {[(words2[h], words2[d]) for h,d in arcs2]}")
print(f"UAS: {score:.3f}")
assert score == 1.0, "Oracle should produce the gold tree"

<details><summary>Solution</summary>

```python
oracle2 = ['SHIFT', 'SHIFT', 'LEFT-ARC',   # she <- ate
           'SHIFT', 'RIGHT-ARC',           # pizza <- ate
           'RIGHT-ARC']                    # ate <- ROOT
arcs2 = arc_standard_parse(words2, oracle2, verbose=False)
assert uas(arcs2, gold2, len(words2)) == 1.0
```

The arc-standard system always reduces the tree bottom-up: dependents must collect all *their* children before being attached to their head (the 'arc-eager' variant relaxes this). A real parser replaces our hand-written oracle with a neural classifier that predicts the next action from the configuration's features.

</details>

## Key takeaways

- **Arc-standard = stack + buffer + 3 transitions,** building a tree in linear time.
- **Well-formed output:** $n-1$ arcs, one head per token (verified).
- **UAS** measures accuracy as the fraction of correct heads (verified).
- **Error propagation:** greedy parsing can't recover from an early mistake (demo) — beam
  search or graph-based parsing helps.